In [ ]:
!pip install langgraph

In [ ]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken

In [8]:
import os
import operator
from typing import TypedDict, Annotated, Optional, get_type_hints
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, ToolMessage,
)
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
llm = ChatOpenAI(model="gpt-4o-mini")

In [6]:
#어제(28일)에 이어서..
class QuestionState(TypedDict):
  question: str
  qtype : str
  answer : str

# 여러 검색 루트를 타고 오게 만들고,
# merge에서 context를 만든 후 LLM으로 전달할 수도 있음
def graph_retrieval(state):
  return {'answer' : '[시간] 답변'}
def vector_retrival(state):
  return {'answer' : '[장소] 답변'}
def h_what(state):
  return {'answer' : '[일반] 답변'}
def h_stmt(state):
  return {'answer' : '[진술문] 답변'}

def merge(state):
  return {'answer' : state['answer'] + '!!!!!'}

def classify(state):
  q = state['question'].lower()
  if '?' not in q: return {'qtype' : 'statement'}
  if any(w in q for w in ['언제', 'when']): return {'qtype': 'when'}
  if any(w in q for w in ['어디', 'where']): return {'qtype': 'where'}
  return {'qtype' : 'what'}

b = StateGraph(QuestionState)
b.add_node('classify', classify)
for name, fn in [('h_when', h_when), ('h_where', h_where), ('h_what', h_what), ('h_stmt', h_stmt)]:
  b.add_node(name, fn)

b.add_node('merge', merge)

b.add_edge(START, 'classify')
b.add_conditional_edges('classify', lambda s: s['qtype'],
                        {'when' : 'h_when','where' : 'h_where', 'what':'h_what', 'statement':'h_stmt'  })

b.add_edge('h_when', 'merge')
b.add_edge('h_where', 'merge')
b.add_edge('h_what', 'merge')
b.add_edge('h_stmt', 'merge')

b.add_edge('merge', END)

app = b.compile()
app.invoke({'question' : "회의 어디서 해?"})

{'question': '회의 어디서 해?', 'qtype': 'where', 'answer': '[장소] 답변!!!!!'}

In [ ]:
# 멀티 홉?
# 멀티 홉(Multi-hop)은 데이터나 신호가 출발지에서 목적지로 갈 때, 직접 연결되지 않고 하나 이상의 중간 노드(라우터, 서버 등)를 거쳐 전달되는 방

# 아래와 같은 질문을 멀티 홉 reasoning이라고 함
  # a는 삼성역 앞에서 아이스크림을 팝니다. : 삼성역
  # b는 선릉역에서 회사에 다닙니다.                  : b
  # 삼성역은 선릉역 옆에 있습니다.         : 선릉역

# a가 아이스크림을 파는곳 근처에서 회사 다니는 사람은?
# -> a가 아이스크림을 파는곳 : 삼성역
  # 삼성역 근처: 선릉역, 역삼역, ...
  # 선릉역에서 회사 다니는 사람: b


In [10]:
# 리듀서(reducer)

def my_reducer(cur, new):
  return (cur or []) + new

class AnnotatedState(TypedDict):
  items : Annotated[list, my_reducer] # Annotated? 앞은 타입, 뒤에는 리듀서 함수를 붙임

hints = get_type_hints(AnnotatedState, include_extras = True)
hints

{'items': typing.Annotated[list, <function my_reducer at 0x7e0dcaaeb420>]}

In [11]:
# Annotated 예시 (원래 사용법)
  # Annotated[list, '설명'] 원래는 메타데이터(설명)을 써줌
  # 랭그래프의 리듀서 함수를 넣어주게 된다면? 리듀서 함수로 해석함


- LangGraph의 기본 동작은 덮어쓰기(overwrite) 입니다. 노드가 {'msgs': ['A 추가']}를 반환하면, 원래 있던 msgs를 그냥 통째로 교체해버려요.
- 그런데 Annotated[list, operator.add]를 붙이면 동작이 바뀝니다:

In [14]:
import operator
class AccumState(TypedDict):
  msgs : Annotated[list, operator.add]

def node_a(state): return {'msgs' :['A 추가']}
def node_b(state): return {'msgs' : ['B 추가']}

b = StateGraph(AccumState)
b.add_node('node_a', node_a)
b.add_node('node_b', node_b)

b.add_edge(START, 'node_a')
b.add_edge('node_a', 'node_b')
b.add_edge('node_b', END)

app = b.compile()
app.invoke({'msgs' : []})

{'msgs': ['A 추가', 'B 추가']}

In [15]:
# operator.add?
    # 인자 (1,2) => 3반환
    # ([1],[2]) => [1,2] 반환

<module 'operator' from '/usr/lib/python3.12/operator.py'>

In [ ]:
# 점수 누적 State 실습
# ScoreState : scores list

# add_kor : scores를 80으로
# add_eng : scores를 90으로
# add_math : scores를 70으로

# 각각 업데이트 해주는 함수

In [16]:
class ScoreState(TypedDict):
  scores : Annotated[list, operator.add]

def add_kor(state):
    return {'scores': [80]}

def add_eng(state):
    return {'scores': [90]}

def add_math(state):
    return {'scores': [70]}

b = StateGraph(ScoreState)
b.add_node('add_kor', add_kor)
b.add_node('add_eng', add_eng)
b.add_node('add_math', add_math)

b.add_edge(START, 'add_kor')
b.add_edge('add_kor', 'add_eng')
b.add_edge('add_eng', 'add_math')
b.add_edge('add_math', END)

app = b.compile()
app.invoke({'scores': []})

{'scores': [80, 90, 70]}

### 메시지 쌓기

In [17]:
from langgraph.graph.message import add_messages

In [22]:
# operator.add / add_messages
class ChatState(TypedDict):
  messages : Annotated[list, add_messages]

def user_says(state):
  return {'messages' : [HumanMessage(content = '안녕')]}

def bot_says(state):
  return {'messages' : [AIMessage(content = '반갑습니다.')]}

b = StateGraph(ChatState)
b.add_node('user_says', user_says)
b.add_node('bot_says', bot_says)

b.add_edge(START, 'user_says')
b.add_edge('user_says', 'bot_says')
b.add_edge('bot_says', END)

app = b.compile()

result = app.invoke({'messages' : []})

result

{'messages': [HumanMessage(content='안녕', additional_kwargs={}, response_metadata={}, id='d6cbd9ba-2a44-439e-ac7e-cf2ee836374c'),
  AIMessage(content='반갑습니다.', additional_kwargs={}, response_metadata={}, id='b04c6661-32ac-4cf0-b618-405b78cf0bdf', tool_calls=[], invalid_tool_calls=[])]}

In [24]:
sys_m = SystemMessage(content = '당신은 컴퓨터 전문가입니다.')
hum_m = HumanMessage(content = '랭그래프?')

ai_m = AIMessage(content = '그래프 기반 라이브러리')

tool_m = ToolMessage(content = '검색결과...', tool_call_id='call_abc123')


In [26]:
for m in [sys_m, hum_m, ai_m, tool_m]:
  print(f"Type = {m.type} | class = {type(m).__name__} | content = {m.content}")

Type = system | class = SystemMessage | content = 당신은 컴퓨터 전문가입니다.
Type = human | class = HumanMessage | content = 랭그래프?
Type = ai | class = AIMessage | content = 그래프 기반 라이브러리
Type = tool | class = ToolMessage | content = 검색결과...


#### add_messages: History를 append()를 이용해 더한 것 처럼...

In [30]:
add_messages(sys_m, hum_m)

[SystemMessage(content='당신은 컴퓨터 전문가입니다.', additional_kwargs={}, response_metadata={}, id='5b37d509-823b-41e9-a652-625ee9a63761'),
 HumanMessage(content='랭그래프?', additional_kwargs={}, response_metadata={}, id='406f7d65-51fe-4182-9a8b-d6cf03909810')]

In [29]:
add_messages(add_messages(sys_m, hum_m), ai_m)

[SystemMessage(content='당신은 컴퓨터 전문가입니다.', additional_kwargs={}, response_metadata={}, id='5b37d509-823b-41e9-a652-625ee9a63761'),
 HumanMessage(content='랭그래프?', additional_kwargs={}, response_metadata={}, id='406f7d65-51fe-4182-9a8b-d6cf03909810'),
 AIMessage(content='그래프 기반 라이브러리', additional_kwargs={}, response_metadata={}, id='c5783592-8157-4633-a729-2c9f690435c4', tool_calls=[], invalid_tool_calls=[])]

In [31]:
ai_with_tools = AIMessage(
    content = "",
    tool_calls = [
        {'name' : 'search', 'args' : {'q' : '랭그래프'}, "id" : "call_001"},
        {'name': 'calc', 'args' : {'x': 7, 'y': 3}, 'id' : 'call_002'}
    ]
)

In [32]:
ai_with_tools.content

''

In [33]:
for tc in ai_with_tools.tool_calls:
  print(f' - {tc['name']} ({tc['args']}), id = {tc['id']}')

 - search ({'q': '랭그래프'}), id = call_001
 - calc ({'x': 7, 'y': 3}), id = call_002


In [34]:
m1 = AIMessage(content = '첫 응답', id = 'msg_x')
m2 = AIMessage(content = '두번째 응답', id = 'msg_x')

merged = add_messages([m1], [m2])
merged # 같은 id('msg_x')를 넣게 되면, 하나(새로운 항목)로 덮어 씌워지는 것을 볼 수 있음

[AIMessage(content='두번째 응답', additional_kwargs={}, response_metadata={}, id='msg_x', tool_calls=[], invalid_tool_calls=[])]

In [35]:
operator.add([1], [2])

[1, 2]

### Re Act(Reasoning + Acting)
- "한번 보고 다시 행동한다"보다는 "생각(Reason)하고 행동(Act)하기를 반복한다" 가 더 정확한 의미입니다. 2022년 프린스턴+구글 논문(Yao et al., "ReAct: Synergizing Reasoning and Acting in Language Models")에서 제안된 방법
- LLM이 한 번에 답을 못 내는 문제를, "생각 → 도구 사용 → 관찰 → 다시 생각" 의 루프로 풀게 하는 패턴

#### 구체적 예시
- LLM이 한 번에 답을 못 내는 문제를, "생각 → 도구 사용 → 관찰 → 다시 생각" 의 루프로 풀게 하는 패턴
- 최대 루프 횟수(예:25) 등을 지정

In [ ]:
# def my_reducer(cur, new):
#   return (cur or []) + new

In [38]:
def keep_max(cur, new): # 10, 30
  return max(cur or 0, new)  # 30

class MaxState(TypedDict):
  score : Annotated[int, keep_max] # 항상 최대값만 유지

def round1(state) : return {'score' : 50}
def round2(state) : return {'score' : 80}
def round3(state) : return {'score' : 70}

b = StateGraph(MaxState)
for n, fn in [('round1', round1), ('round2', round2), ('round3', round3)]:
  b.add_node(n,fn)

b.add_edge(START, 'round1')
b.add_edge('round1', 'round2')
b.add_edge('round2', 'round3')
b.add_edge('round3', END)

app = b.compile()
app.invoke({'score' : 0})



{'score': 80}

In [43]:
# add int -> 새 값이 들어오면 플러스 10, 20 -> 30
# SumState : total(int)

# s1: return {'total' : 10}
# s2 : 20
# s3 : 30

# app.invoke({'total} : 0)

def add_int(cur, new):
  return new + cur

class SumState(TypedDict):
  total : Annotated[int, add_int]

def round1(state) : return {'total' : 30}
def round2(state) : return {'total' : 20}
def round3(state) : return {'total' : 10}

b = StateGraph(SumState)
for n, fn in [('round1', round1), ('round2', round2), ('round3', round3)]:
  b.add_node(n,fn)

b.add_edge(START, 'round1')
b.add_edge('round1', 'round2')
b.add_edge('round2', 'round3')
b.add_edge('round3', END)

app = b.compile()
app.invoke({'total' : 0})

{'total': 60}

In [44]:
def merge_dicts(cur, new):
  return {**(cur or {}), **(new or {})}


class MetaState(TypedDict):
  meta : Annotated[dict, merge_dicts]

def fetch_user(state) : return {'meta' : {'user_id' : 'u123'}}
def fetch_session(state) : return {'meta' : {'session_id' : 's456'}}
def fetch_locale(state) : return {'meta' : {'locale' : 'ko_KR'}}


b = StateGraph(MetaState)
for n, fn in [('fetch_user', fetch_user), ('fetch_session', fetch_session), ('fetch_locale', fetch_locale)]:
  b.add_node(n,fn)

b.add_edge(START, 'fetch_user')
b.add_edge('fetch_user', 'fetch_session')
b.add_edge('fetch_session', 'fetch_locale')
b.add_edge('fetch_locale', END)


app = b.compile()
app.invoke({'meta' : {}})


{'meta': {'user_id': 'u123', 'session_id': 's456', 'locale': 'ko_KR'}}

In [52]:
# ConfigState(TypedDict):
#   config(dict)

# set_model       model          gpt-4o-mini
# set_temperature temperature    2.0
# override_model  model          gpt-4o

def merge_dicts(cur, new):
  return {**(cur or {}), **(new or {})}

class ConfigState(TypedDict):
  config : Annotated[dict, merge_dicts]

def set_model(state) : return {'config' : {'model' : 'gpt-4o-mini'}}
def set_temperature(state) : return {'config' : {'temperature' : 2.0}}
def override_model(state) : return {'config' : {'model' : 'gpt-4o'}}

b = StateGraph(ConfigState)
for n, fn in [('set_model', set_model), ('set_temperature', set_temperature), ('override_model', override_model)]:
  b.add_node(n,fn)

b.add_edge(START, 'set_model')
b.add_edge('set_model', 'set_temperature')
b.add_edge('set_temperature', 'override_model')
b.add_edge('override_model', END)

app = b.compile()
app.invoke({'config' : {}})

{'config': {'model': 'gpt-4o', 'temperature': 2.0}}

### 멀티 채널 스테이트
- 채널 별로 다른 리듀서를 가지는 것
- 각각의 노드들은 자기와 관련된 채널(키)만 업데이트

In [53]:
class MultiState(TypedDict):
  messages : Annotated[list, add_messages]
  score : Annotated[int, keep_max]
  meta : Annotated[dict, merge_dicts]
  raw : str

def n1(state):
  return {
      'messages' : [HumanMessage(content= '안녕')],
      'score' : 50,
      'meta' : {"step": 1},
      'raw' : "first"
  }

def n2(state):
  return {
      'messages' : [AIMessage(content= '반가워')],
      'score' : 60,
      'meta' : {"step": 2},
      'raw' : "second"
  }

b = StateGraph(MultiState)
b.add_node('n1', n1)
b.add_node('n2', n2)
b.add_edge(START, 'n1')
b.add_edge('n1', 'n2')
b.add_edge('n2', END)

app = b.compile()

result = app.invoke({'messages' : [], 'score':0, 'meta':{}, 'raw' : ''})
display(result)

{'messages': [HumanMessage(content='안녕', additional_kwargs={}, response_metadata={}, id='a7b4b03e-72fd-4b2c-8fc4-a7c4719ee48f'),
  AIMessage(content='반가워', additional_kwargs={}, response_metadata={}, id='1228f737-710f-4ed3-b244-1a8397bb2e39', tool_calls=[], invalid_tool_calls=[])],
 'score': 60,
 'meta': {'step': 2},
 'raw': 'second'}

In [54]:
len(result['messages'])

2

In [58]:
# GameState
# events : list -> 이벤트 로그를 출력
# best : int -> 최고 점수
# stats: dict -> merge_dicts로 업데이트

# r1: events = ['round1'], best = 30, stats = {'plays':1}
# r2: events = ['round2'], best = 50, stats = {'wins':1}

def keep_max(cur, new):
    return max(cur or 0, new or 0)

def merge_dicts(cur, new):
    return {**(cur or {}), **(new or {})}

class GameState(TypedDict):
    events: Annotated[list, operator.add] # 이벤트 로그 출력용
    best: Annotated[int, keep_max]
    stats: Annotated[dict, merge_dicts]

def r1(state):
    return {
        'events': ['round1'],
        'best': 30,
        'stats': {'plays': 1}
    }

def r2(state):
    return {
        'events': ['round2'],
        'best': 50,
        'stats': {'wins': 1}
    }

b = StateGraph(GameState)
b.add_node('r1', r1)
b.add_node('r2', r2)

b.add_edge(START, 'r1')
b.add_edge('r1', 'r2')
b.add_edge('r2', END)

app = b.compile()

result = app.invoke({'events': [], 'best': 0, 'stats': {}})
print(result)

{'events': ['round1', 'round2'], 'best': 50, 'stats': {'plays': 1, 'wins': 1}}


In [60]:
# 컨텍스트 자르기 (최근 n개 메시지 자르기)

class TrimState(TypedDict):
  messages : Annotated[list, add_messages]
  visible : list

N = 3

def add_many(state):
  new_msgs = [HumanMessage(content = f'msg-{i}') for i in range (10)]
  return {'messages' : new_msgs}

def trim_node(state):
  return {'visible' : state['messages'][-N:]}

b = StateGraph(TrimState)
b.add_node('add_many', add_many)
b.add_node('trim_node', trim_node)
b.add_edge(START, 'add_many')
b.add_edge('add_many', 'trim_node')
b.add_edge('trim_node', END)

app = b.compile()

result = app.invoke({'messages' : [], 'visible': []})
result # visible에는 가장 최근에 들어간 상위 3개 메시지만 들어가있는것을 확인 가능

{'messages': [HumanMessage(content='msg-0', additional_kwargs={}, response_metadata={}, id='bb9ff914-aaf3-45f6-95ff-ffea5248aa7f'),
  HumanMessage(content='msg-1', additional_kwargs={}, response_metadata={}, id='32bc451e-acb7-4a03-a98c-29e99a3ec942'),
  HumanMessage(content='msg-2', additional_kwargs={}, response_metadata={}, id='65368a49-52ec-4338-a735-aefd5d1b5432'),
  HumanMessage(content='msg-3', additional_kwargs={}, response_metadata={}, id='a3c7c68a-b649-4444-9563-9176b7c35bad'),
  HumanMessage(content='msg-4', additional_kwargs={}, response_metadata={}, id='6c89c2c7-e046-4768-b7c4-9e638db6cf76'),
  HumanMessage(content='msg-5', additional_kwargs={}, response_metadata={}, id='c08695d5-800b-4306-82aa-473a69f901fa'),
  HumanMessage(content='msg-6', additional_kwargs={}, response_metadata={}, id='b30779d4-e390-4f48-8312-2889c5d45434'),
  HumanMessage(content='msg-7', additional_kwargs={}, response_metadata={}, id='e97069f4-3d24-49c4-bbdc-eb4e9bbee3b1'),
  HumanMessage(content='msg-

In [61]:
from langgraph.graph.message import RemoveMessage

In [62]:
# RemoveMessage 활용
  # 컨텍스트 자르기 (최근 n개 메시지 자르기)

class TrimState(TypedDict):
  messages : Annotated[list, add_messages]

N = 3

def add_many(state):
  new_msgs = [HumanMessage(content = f'msg-{i}') for i in range (10)]
  return {'messages' : new_msgs}

def trim_node(state):
  msgs = state['messages']
  to_remove = msgs[:-N]
  return {'messages' :[RemoveMessage(id = m.id) for m in to_remove]}

b = StateGraph(TrimState)
b.add_node('add_many', add_many)
b.add_node('trim_node', trim_node)
b.add_edge(START, 'add_many')
b.add_edge('add_many', 'trim_node')
b.add_edge('trim_node', END)

app = b.compile()

result = app.invoke({'messages' : []})
result

{'messages': [HumanMessage(content='msg-7', additional_kwargs={}, response_metadata={}, id='f3a105e7-89dc-46bb-9311-88d6c39e998a'),
  HumanMessage(content='msg-8', additional_kwargs={}, response_metadata={}, id='6bb32136-3528-4b88-8104-d9b3e56fe092'),
  HumanMessage(content='msg-9', additional_kwargs={}, response_metadata={}, id='fec25e21-e84d-4964-aac4-e30ca6b70b1e')]}

In [64]:
# 툴 메시지는 빼고 싶은 경우
# 휴먼 메시지만 보내고 싶은 경우..
# 함수 하나 하나는 state

mixed = [
    SystemMessage(content = '시스템'),
    HumanMessage(content = '안녕'),
    AIMessage(content = '반가워'),
    ToolMessage(content = '검색 결과', tool_call_id = 't1'), # 쉼표 추가
    HumanMessage(content = 'LangGraph')
]

def only_human(msgs):
    return [m for m in msgs if isinstance(m, HumanMessage)]

def drop_tool(msgs):
    return [m for m in msgs if not isinstance(m, ToolMessage)]

display(drop_tool(mixed))

[SystemMessage(content='시스템', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='안녕', additional_kwargs={}, response_metadata={}),
 AIMessage(content='반가워', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='LangGraph', additional_kwargs={}, response_metadata={})]

In [65]:
# 실습
# FilterState : messages (list), reply (str)

# seed : return {'messages': [
#     SystemMessage(content = '시스템'),
#     HumanMessage(content = '안녕'),
#     AIMessage(content = '반가워'),
#     ToolMessage(content = '검색 결과', tool_call_id = 't1'), # 쉼표 추가
#     HumanMessage(content = 'LangGraph')]}

# respond :
#   tool message drop
#   response = llm.invoke(dropped message)
#   return {'reply' : response.content}

class FilterState(TypedDict):
  messages: list
  reply : str

def seed(state):
    return {
        'messages': [
            SystemMessage(content='시스템'),
            HumanMessage(content='안녕'),
            AIMessage(content='반가워'),
            ToolMessage(content='검색 결과', tool_call_id='t1'),
            HumanMessage(content='LangGraph'),
        ]
    }

def respond(state):
    dropped = [m for m in state['messages'] if not isinstance(m, ToolMessage)]
    response = llm.invoke(dropped)
    return {'reply': response.content}


b = StateGraph(FilterState)
b.add_node('seed', seed)
b.add_node('respond', respond)
b.add_edge(START, 'seed')
b.add_edge('seed', 'respond')
b.add_edge('respond', END)

app = b.compile()

result = app.invoke({'messages' : []})



In [66]:
result

{'messages': [SystemMessage(content='시스템', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='안녕', additional_kwargs={}, response_metadata={}),
  AIMessage(content='반가워', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  ToolMessage(content='검색 결과', tool_call_id='t1'),
  HumanMessage(content='LangGraph', additional_kwargs={}, response_metadata={})],
 'reply': '"LangGraph"라는 용어는 특정한 소프트웨어, 라이브러리, 또는 개념을 의미할 수 있지만, 구체적인 정보가 제공되지 않으면 정확한 설명을 드리기 어렵습니다. LangGraph가 어떤 맥락에서 사용되는지, 혹은 어떤 기능이나 목적을 가지고 있는지 구체적으로 말씀해 주시면 더 자세히 설명해 드릴 수 있습니다.'}

In [69]:
class TurnState(TypedDict):
  messages : Annotated[list, add_messages]

def chat_turn(state):
  response = llm.invoke(state['messages'])
  return {'messages' : [response]}

b = StateGraph(TurnState)

b.add_node('chat_turn', chat_turn)
b.add_edge(START, 'chat_turn')
b.add_edge('chat_turn', END)

turn_app = b.compile()

history = [SystemMessage(content = '당신은 짧게 대답하는 도우미입니다.')]
for user_text in ['안녕!', '내 이름은 ㅇㅇㅇ야', '내 이름이 뭐였지?']:
  out = turn_app.invoke({'messages' : history + [HumanMessage(content = user_text)]})
  history = out['messages']
  print(f"USER : {user_text}")
  print(f"BOT : {history[-1].content}" )

USER : 안녕!
BOT : 안녕하세요! 무엇을 도와드릴까요?
USER : 내 이름은 ㅇㅇㅇ야
BOT : 만나서 반가워, ㅇㅇㅇ! 어떻게 도와드릴까요?
USER : 내 이름이 뭐였지?
BOT : 당신의 이름은 ㅇㅇㅇ입니다!
